<a href="https://colab.research.google.com/github/vad-source/AIMLROBOTICS/blob/main/MeasurementModel/NavBot_Measurement_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## AIMLROBOTICS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

**Objective:**
>> To familiarize with the notion of measurement model and its gaussian noise

>> Assume a mobile robot is moving along a straight line in a hallway for 10 steps, sensing a wall in front, correcting the measurement errors.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from time import sleep

In [ ]:
wall_position = 10.0   # Wall at 10m
sigma = 0.2            # To simulate the behaviour of incorrect measurement this sensor noise std is added for demo purposes

# Robot positions (10 steps)
robot_positions = np.linspace(1, 9, 10)

observed_measurements = []
expected_measurements = []
corrected_measurements = []

# Simple correction: exponential smoothing
alpha = 0.6

In [ ]:
plt.figure(figsize=(10, 5))

for i, x in enumerate(robot_positions):

    plt.clf()

    # -----------------------------
    # 1. Expected Measurement (Ray Casting)
    # -----------------------------
    Z_expected = wall_position - x

    # -----------------------------
    # 2. Simulated Sensor Reading
    # -----------------------------
    noise = np.random.normal(0, sigma)
    Z_measured = Z_expected + noise

    # -----------------------------
    # 3. Likelihood (Gaussian)
    # -----------------------------
    likelihood = norm.pdf(Z_measured - Z_expected, 0, sigma)

    # -----------------------------
    # 4. Corrected Estimate (Smoothed)
    # -----------------------------
    if i == 0:
        Z_corrected = Z_measured
    else:
        Z_corrected = alpha * Z_measured + (1 - alpha) * corrected_measurements[-1]

    # Store values
    expected_measurements.append(Z_expected)
    observed_measurements.append(Z_measured)
    corrected_measurements.append(Z_corrected)

    # -----------------------------
    # PLOT 1: ENVIRONMENT VIEW
    # -----------------------------
    plt.subplot(1, 2, 1)
    plt.title(f"Step {i+1}: Robot in Hallway")

    # Wall
    plt.plot([wall_position, wall_position], [0, 1], 'k-', linewidth=4, label="Wall")

    # Robot
    plt.scatter(x, 0.5, c='blue', s=100, label="Robot")

    # Expected beam (green)
    plt.plot([x, wall_position], [0.5, 0.5],
             'g-', linewidth=2, label="Expected Beam (Z*)")

    # Observed beam (red)
    plt.plot([x, x + Z_measured], [0.5, 0.7],
             'r--', linewidth=2, label="Observed (Z)")

    # Corrected beam (purple)
    plt.plot([x, x + Z_corrected], [0.5, 0.3],
             'm-.', linewidth=2, label="Corrected")

    plt.xlim(0, 11)
    plt.ylim(0, 1)
    plt.xlabel("Distance (m)")
    plt.yticks([])
    plt.legend(loc='upper left')

    # -----------------------------
    # PLOT 2: MEASUREMENT GRAPH
    # -----------------------------
    plt.subplot(1, 2, 2)
    plt.title("Measurements Over Steps")

    steps = np.arange(1, i+2)

    plt.plot(steps, expected_measurements, 'g-o', label="Expected (Z*)")
    plt.plot(steps, observed_measurements, 'r-o', label="Observed (Z)")
    plt.plot(steps, corrected_measurements, 'm-o', label="Corrected")

    plt.xlabel("Step")
    plt.ylabel("Distance (m)")
    plt.legend()
    plt.grid(True)

    # -----------------------------
    # PRINT NUMERIC VALUES
    # -----------------------------
    print(f"Step {i+1}")
    print(f"Position X = {x:.2f}")
    print(f"Expected Z* = {Z_expected:.2f}")
    print(f"Measured Z = {Z_measured:.2f}")
    print(f"Corrected Z = {Z_corrected:.2f}")
    print(f"Likelihood P(Z|X) = {likelihood:.3f}")
    print("-" * 40)

    plt.tight_layout()
    plt.pause(1.2)

plt.show()
